# ETS MARL — Bots-Only Baseline

This notebook runs and analyses a **heuristic-bots-only** baseline for the EU ETS multi-agent simulation: 8 rule-based bots (`src/agents/heuristic_policy.py`), zero learning agents. It is the *no-learning* counterpart to:

* `ets_marl - Full Run & Analysis.ipynb` — full PPO/HAPPO training + analysis
* `ets_marl - Q-Learning Baseline.ipynb` — tabular Q-learning baseline + cross-comparison

Together they bracket the agent-space comparison: bots ≤ Q-learning ≤ PPO/HAPPO. The bots-only floor tells us what an *un-trained, fundamentals-driven* market looks like under the same default-config calibration — any uplift from the learners has to clear this bar to count as evidence the agent space is doing useful work.

**Sections.**

1. Setup
2. Configuration
3. One demo episode (sanity check)
4. Multi-seed training run (`scripts/run_bots_only.py`)
5. Cross-seed market trajectories (price, compliance, green frac, quality)
6. Per-bot performance summary
7. Comparison vs Q-learning + PPO sweep (when logs exist)
8. Takeaways

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
import copy
import glob
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.environment.ets_environment import ETSEnvironment
from scripts.run_bots_only import run_bots_only_seed

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams['figure.dpi'] = 110

## 2. Configuration

Load the bots-only config (`configs/bots_only.yaml`). It inherits the production calibration of `configs/default.yaml` and only overrides participant counts (`n_agents=0`, `n_bot_agents=8`) and the bot-specific budget/headroom mirrors.

In [ ]:
with open('configs/bots_only.yaml') as f:
    config = yaml.safe_load(f)

n_agents = int(config['companies']['n_agents'])
n_bots = int(config['companies']['n_bot_agents'])
n_total = n_agents + n_bots
n_years = int(config['simulation']['n_years'])

print(f'Learning agents : {n_agents}')
print(f'Heuristic bots  : {n_bots}')
print(f'Total participants: {n_total}')
print(f'Episode length   : {n_years} years')
print(f'Seeds            : {config["simulation"]["seeds"]}')
print(f'Episodes / seed  : {config["simulation"]["n_episodes"]}')

## 3. One demo episode (sanity check)

Before the full sweep, run a single deterministic-seed episode and look at the year-by-year trajectory. The bots should be roughly compliant (low shortfalls), prices should rise as the cap tightens, and green share should grow over the 12-year horizon.

In [ ]:
env = ETSEnvironment(config, seed=42)
env.reset(seed=42)
no_auc = np.zeros((0, 6), dtype=np.float32)
no_sec = np.zeros((0, 2), dtype=np.float32)

rows = []
for y in range(n_years):
    env.step_auction(no_auc)
    _, _, _, _, info = env.step_secondary(no_sec)
    log = info['year_log']
    rows.append({
        'year': y + 1,
        'cap': log['cap'],
        'tnac': log['tnac'],
        'auction_volume': log['auction_volume'],
        'clearing_price': log['clearing_price'],
        'mean_green_frac': float(np.mean(log['green_fracs'])),
        'compliant_agents': int(sum(s <= 1e-6 for s in log['shortfalls'])),
        'mean_shortfall': float(np.mean(log['shortfalls'])),
    })

demo_df = pd.DataFrame(rows)
demo_df.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ax = axes[0, 0]
ax.plot(demo_df['year'], demo_df['clearing_price'], 'o-')
ax.set_title('Auction clearing price'); ax.set_xlabel('year'); ax.set_ylabel('EUR/t')
ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(demo_df['year'], demo_df['cap'], 'o-', label='cap')
ax.plot(demo_df['year'], demo_df['tnac'], 's-', label='TNAC', alpha=0.7)
ax.plot(demo_df['year'], demo_df['auction_volume'], '^-', label='auction vol', alpha=0.7)
ax.set_title('Cap / TNAC / auction volume'); ax.set_xlabel('year'); ax.set_ylabel('Mt')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(demo_df['year'], demo_df['mean_green_frac'] * 100, 'o-', color='green')
ax.set_title('Mean green fraction (8 bots)'); ax.set_xlabel('year'); ax.set_ylabel('%')
ax.set_ylim(0, 100); ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.bar(demo_df['year'], demo_df['compliant_agents'], color='steelblue')
ax.axhline(n_bots, ls='--', color='gray', alpha=0.6, label=f'all {n_bots} compliant')
ax.set_title('Compliant agents per year'); ax.set_xlabel('year'); ax.set_ylabel('count')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## 4. Multi-seed run

Run the heuristic market for many episodes per seed. Each episode has fresh stochasticity (per-bot valuation noise, urgency multipliers, demand shocks, construction failures), so cross-episode means describe the steady-state market. Logs are written to `results/bots_only/`.

In [ ]:
SEEDS = list(config['simulation']['seeds'])
N_EPISODES = int(config['simulation']['n_episodes'])
OUTPUT_DIR = 'results/bots_only/'

# Bots-only is fast (~1ms / episode); the default 500 episodes × 3 seeds runs in seconds.
# To re-use existing logs without re-running, set RERUN = False.
RERUN = True

if RERUN:
    for seed in SEEDS:
        run_bots_only_seed(
            config=config, seed=seed,
            n_episodes=N_EPISODES,
            output_dir=OUTPUT_DIR,
            run_tag=None,
            log_interval=max(1, N_EPISODES // 10),
        )

print('\nLogs written to:', OUTPUT_DIR)

In [ ]:
# Load all per-seed training logs into a single long frame.
def _load_seed_log(path: str, seed: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['seed'] = seed
    return df

train_dfs, year_dfs = [], []
for seed in SEEDS:
    tp = os.path.join(OUTPUT_DIR, f'training_log_s{seed}.csv')
    yp = os.path.join(OUTPUT_DIR, f'year_log_s{seed}.csv')
    if os.path.exists(tp):
        train_dfs.append(_load_seed_log(tp, seed))
    if os.path.exists(yp):
        year_dfs.append(_load_seed_log(yp, seed))

train_df = pd.concat(train_dfs, ignore_index=True)
year_df = pd.concat(year_dfs, ignore_index=True)
print(f'train_df: {len(train_df):,} rows ({train_df["seed"].nunique()} seeds)')
print(f'year_df : {len(year_df):,} rows')
train_df.head(3)

## 5. Cross-seed market trajectories

For each seed, plot the per-episode market metrics. Heuristic bots don't *learn*, so the curves should be flat-ish bands of stochastic variation — the run-to-run shape is set by the production calibration (cap path, MSR, penalties, ESG), not by training.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics = [
    ('ep_mean_clearing_price', 'Episode-mean clearing price (EUR/t)'),
    ('compliance_rate', 'Compliance rate (fraction of agent-years)'),
    ('mean_green_frac', 'End-of-episode mean green fraction'),
    ('quality_score', 'Quality score (signed, [-5, +5])'),
]
for ax, (col, title) in zip(axes.flat, metrics):
    for seed in SEEDS:
        sd = train_df[train_df['seed'] == seed]
        if col not in sd.columns:
            continue
        ax.plot(sd['episode'], sd[col].rolling(window=max(5, len(sd) // 20), min_periods=1).mean(),
                label=f'seed={seed}', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('episode')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Year-of-episode price profile, averaged across all (episode, seed)
year_price = year_df.groupby('year')['clearing_price'].agg(['mean', 'std']).reset_index()
year_green = year_df.groupby('year')[[f'green_frac_A{i+1}' for i in range(n_total)]].mean().mean(axis=1).reset_index()
year_green.columns = ['year', 'mean_green_frac']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.plot(year_price['year'] + 1, year_price['mean'], 'o-', color='C0')
ax.fill_between(year_price['year'] + 1,
                year_price['mean'] - year_price['std'],
                year_price['mean'] + year_price['std'],
                color='C0', alpha=0.15)
ax.set_title('Mean clearing price by year-of-episode')
ax.set_xlabel('year'); ax.set_ylabel('EUR/t'); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(year_green['year'] + 1, year_green['mean_green_frac'] * 100, 'o-', color='green')
ax.set_title('Mean green fraction by year-of-episode')
ax.set_xlabel('year'); ax.set_ylabel('%'); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
plt.tight_layout()

## 6. Per-bot performance

Summary table over the last 25% of episodes per seed, then averaged across seeds. The four archetype pairs (coal-heavy, gas-dominant, transitioner, green-leader) split into financial / ESG variants by reward weights — financial bots minimise costs, ESG bots also pay attention to green share.

In [ ]:
tail_frac = 0.25
def _last_window(df, frac=tail_frac):
    cutoff = int(len(df) * (1 - frac))
    return df.iloc[cutoff:]

archetypes = ['coal-h/fin', 'coal-h/ESG', 'gas-d/fin', 'gas-d/ESG',
              'trans/fin', 'trans/ESG', 'green-l/fin', 'green-l/ESG']

rows = []
for i in range(n_total):
    name = archetypes[i] if i < len(archetypes) else f'B{i+1}'
    row = {'bot': f'B{i+1}', 'archetype': name}
    for col, label in [('reward', 'reward'),
                       ('green_frac', 'green%'),
                       ('shortfall', 'shortfall'),
                       ('penalty', 'penalty (M€)'),
                       ('compliance_rate', 'comply%')]:
        c = f'{col}_A{i+1}'
        if c not in train_df.columns:
            row[label] = np.nan; continue
        per_seed = []
        for seed in SEEDS:
            sd = _last_window(train_df[train_df['seed'] == seed])
            per_seed.append(sd[c].mean())
        v = float(np.mean(per_seed))
        if col in ('green_frac', 'compliance_rate'):
            v *= 100
        row[label] = round(v, 3)
    rows.append(row)

summary = pd.DataFrame(rows)
summary

## 7. Comparison with Q-learning and PPO sweep (optional)

If the Q-learning baseline (`results/qlearning/ql_training_log_s*.csv`) and/or the default-config PPO sweep (`results/sweep_default_seeds/.../training_log_s*.csv`) have been run, overlay their convergence curves so the bots-only floor sits in context.

In [ ]:
# Q-learning logs
ql_paths = sorted(glob.glob('results/qlearning/ql_training_log_s*.csv'))
ql_dfs = []
for p in ql_paths:
    try:
        seed = int(p.split('_s')[-1].split('.csv')[0])
    except ValueError:
        continue
    df = pd.read_csv(p); df['seed'] = seed
    ql_dfs.append(df)
ql_df = pd.concat(ql_dfs, ignore_index=True) if ql_dfs else None
print(f'Q-learning logs: {len(ql_paths)} seed(s)' if ql_df is not None else 'Q-learning: not found')

# PPO default-seed sweep
ppo_paths = sorted(glob.glob('results/sweep_default_seeds/*/training_log_s*.csv'))
if not ppo_paths:
    ppo_paths = sorted(glob.glob('results/training_log_s*.csv'))
ppo_dfs = []
for p in ppo_paths:
    try:
        seed = int(p.rsplit('_s', 1)[-1].split('.csv')[0])
    except ValueError:
        continue
    df = pd.read_csv(p); df['seed'] = seed
    ppo_dfs.append(df)
ppo_df = pd.concat(ppo_dfs, ignore_index=True) if ppo_dfs else None
print(f'PPO logs: {len(ppo_paths)} seed(s)' if ppo_df is not None else 'PPO: not found')

In [ ]:
def _seed_band(df, col, win=50):
    out = []
    for seed in df['seed'].unique():
        sd = df[df['seed'] == seed].sort_values('episode')
        out.append(sd[col].rolling(window=win, min_periods=1).mean().reset_index(drop=True))
    if not out:
        return None
    M = pd.concat(out, axis=1)
    return pd.DataFrame({'mean': M.mean(axis=1), 'std': M.std(axis=1)})

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
panels = [
    ('ep_mean_clearing_price', 'Mean clearing price (EUR/t)'),
    ('compliance_rate',        'Compliance rate'),
    ('mean_green_frac',        'Mean green fraction'),
    ('quality_score',          'Quality score'),
]

for ax, (col, title) in zip(axes.flat, panels):
    bot_band = _seed_band(train_df, col, win=max(5, N_EPISODES // 20))
    if bot_band is not None:
        x = np.arange(len(bot_band))
        ax.plot(x, bot_band['mean'], color='C2', label='Bots-only', lw=2)
        ax.fill_between(x, bot_band['mean'] - bot_band['std'],
                        bot_band['mean'] + bot_band['std'], color='C2', alpha=0.15)
    if ql_df is not None and col in ql_df.columns:
        b = _seed_band(ql_df, col)
        if b is not None:
            x = np.arange(len(b))
            ax.plot(x, b['mean'], color='C1', label='Q-learning', lw=1.5)
            ax.fill_between(x, b['mean'] - b['std'], b['mean'] + b['std'], color='C1', alpha=0.15)
    if ppo_df is not None and col in ppo_df.columns:
        b = _seed_band(ppo_df, col, win=200)
        if b is not None:
            x = np.arange(len(b))
            ax.plot(x, b['mean'], color='C0', label='PPO/HAPPO', lw=1.5)
            ax.fill_between(x, b['mean'] - b['std'], b['mean'] + b['std'], color='C0', alpha=0.15)
    ax.set_title(title); ax.set_xlabel('episode'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Converged-window comparison table (last 10% of episodes per source).
def _converged_summary(df, label, cols=('ep_mean_clearing_price', 'compliance_rate',
                                         'mean_green_frac', 'quality_score')):
    rows = []
    for seed in df['seed'].unique():
        sd = df[df['seed'] == seed].sort_values('episode')
        tail = sd.iloc[int(len(sd) * 0.9):]
        rows.append({'source': label, 'seed': seed,
                     **{c: tail[c].mean() if c in tail.columns else np.nan for c in cols}})
    return pd.DataFrame(rows)

frames = [_converged_summary(train_df, 'Bots-only')]
if ql_df is not None:
    frames.append(_converged_summary(ql_df, 'Q-learning'))
if ppo_df is not None:
    frames.append(_converged_summary(ppo_df, 'PPO/HAPPO'))

cmp = pd.concat(frames, ignore_index=True)
cmp_agg = cmp.groupby('source').agg(['mean', 'std']).round(3)
cmp_agg

## 8. Takeaways

* **Bots-only as a credibility floor.** The heuristic-bots-only market is fully rule-based: each bot's bid is a willingness-to-pay between MAC and the inflation-adjusted penalty cap, with NPV-gated investment and budget-headroom-aware secondary trading. There is no learning. Cross-seed quality bands describe the *production calibration on its own*, before any agent gets to optimise against it.

* **Comparison logic.**
  * If Q-learning ≈ bots → the discrete coordination problem is too easy for tabular methods to differentiate from rules.
  * If PPO/HAPPO ≈ bots → the simulation isn't pushing learners to find non-trivial strategies (calibration / reward-shape problem).
  * If PPO/HAPPO > Q-learning > bots → the agent space is doing useful work; the gaps measure exactly how much.

* **What the bots are good at:** compliance under default-config calibration, sensible price discovery (clearing tracks fundamentals), and a moderate green transition (~70% by year 12).

* **What the bots are bad at:** strategic banking, secondary-market timing (the heuristic is fundamentals-only), and exploiting cap-scarcity windfalls. These are the strategy gaps the learning agents are expected to close.